# Первичный анализ

In [1]:
import pandas as pd

base = "https://python-academy.org/static/product-analytics/dataset"


# parse_dates преобразует столбцы в тип даты для датафрейма
users = pd.read_csv(f"{base}/users.csv", parse_dates=["signup_date"])
products = pd.read_csv(f"{base}/products.csv")
sessions = pd.read_csv(f"{base}/sessions.csv", parse_dates=["started_at"])
orders = pd.read_csv(f"{base}/orders.csv", parse_dates=["created_at"])

print(f"users:    {len(users):>7,}")

print(f"products: {len(products):>7,}")

print(f"sessions: {len(sessions):>7,}")

print(f"orders:   {len(orders):>7,}")

users:      6,000
products:     300
sessions:  68,668
orders:     7,327


In [5]:
# Посмотреть тип данных колонок в датафрейме
users.dtypes

user_id                     object
signup_date         datetime64[ns]
source                      object
country                     object
device_first                object
age_bucket                  object
marketing_opt_in              bool
dtype: object

In [7]:
# Посмотреть историчность данных во всех таблицах
print(f"users:    {users['signup_date'].min().date()} → {users['signup_date'].max().date()}")

print(f"sessions: {sessions['started_at'].min().date()} → {sessions['started_at'].max().date()}")

print(f"orders:   {orders['created_at'].min().date()} → {orders['created_at'].max().date()}")

users:    2025-06-01 → 2026-05-31
sessions: 2025-06-01 → 2026-05-31
orders:   2025-06-01 → 2026-06-01


In [12]:
# Проверка пропусков
print(users.isna().sum())

user_id               0
signup_date           0
source                0
country               0
device_first          0
age_bucket          309
marketing_opt_in      0
dtype: int64


In [13]:
print(sessions.isna().sum())

session_id         0
user_id            0
started_at         0
duration_sec       0
device             0
source             0
country         2078
pages_viewed       0
reached_step       0
dtype: int64


In [25]:
# Уникальные значения в колонке
print(users["source"].unique())

['paid_social' 'organic' 'paid_search' 'referral' 'email']


In [14]:
# Проверка дублей
print(f"дублей user_id:    {users['user_id'].duplicated().sum()}")

print(f"дублей session_id: {sessions['session_id'].duplicated().sum()}")

print(f"дублей order_id:   {orders['order_id'].duplicated().sum()}")

дублей user_id:    0
дублей session_id: 8
дублей order_id:   0


In [17]:
# Описание базовой статистики конкретного столбца

# Среднее в полтора раза больше медианы, а максимум в пять раз выше 75-го перцентиля — 
# это намёк на длинный хвост: небольшая доля крупных заказов сильно тянет среднее вверх.
print(orders["total_rub"].describe())

count      7327.000000
mean      34012.965743
std       34653.869367
min         230.000000
25%        8650.000000
50%       21900.000000
75%       47100.000000
max      243400.000000
Name: total_rub, dtype: float64


# Срезы и фильтры

In [19]:
# Конструкция-аналог where

completed = orders[orders["status"] == "completed"]
print(f"завершённых заказов: {len(completed)} из {len(orders)}")

завершённых заказов: 6986 из 7327


In [21]:
# Применение метода к маске считается автоматически по True условию
# Одна строка вместо двух подсчётов и деления — так доли считают в реальной работе

print(f"доля завершённых: {(orders['status'] == 'completed').mean():.1%}")

print(f"доля мобильных сессий: {(sessions['device'] == 'mobile').mean():.1%}")

доля завершённых: 95.3%
доля мобильных сессий: 57.9%


In [24]:
# Конструкция-аналог where с несколькими условиями
# Кроме & есть | (или) и ~ (не)

march = orders[
    (orders["created_at"] >= "2026-03-01") & (orders["created_at"] < "2026-04-01")
]
print(f"заказов в марте: {len(march)}")

заказов в марте: 1025


In [26]:
# Конструкция-аналог in

cis = users[users["country"].isin(["KZ", "BY", "UA"])]
print(f"пользователей из KZ, BY, UA: {len(cis)}")

пользователей из KZ, BY, UA: 1835


In [28]:
# в query можно через строку вставить много условий
# имена колонок без кавычек, а строковые значения — в одинарных кавычках

big = orders.query("total_rub > 50_000 and status == 'completed'")
print(f"крупных завершённых заказов: {len(big)}")

крупных завершённых заказов: 1639


In [30]:
# loc для фильтрации только нужных столбцов датафрейма

kz = users.loc[users["country"] == "KZ", ["user_id", "signup_date", "source"]]
print(f"пользователей из KZ: {len(kz)}")

print(kz.head())

пользователей из KZ: 921
    user_id signup_date       source
0   u_00001  2026-04-04  paid_social
18  u_00019  2026-04-18  paid_search
25  u_00026  2025-09-17     referral
29  u_00030  2026-03-11      organic
31  u_00032  2026-05-23  paid_social


In [31]:
# Для создания нового отфильтрованного датафрейма стоит создать копию среза

kz = users.loc[users["country"] == "KZ", ["user_id", "signup_date", "source"]].copy()

# Группировки

In [3]:
# Сколько строк в каждой группе

print(users["source"].value_counts())

source
paid_social    2082
organic        1502
paid_search    1243
email           594
referral        579
Name: count, dtype: int64


In [5]:
# С аргументом normalize подсчет доли вместо количества

print((users["source"].value_counts(normalize=True) * 100).round(1))

source
paid_social    34.7
organic        25.0
paid_search    20.7
email           9.9
referral        9.6
Name: proportion, dtype: float64


In [6]:
# groupby среднее total_rub по полю items_count

print(orders.groupby("items_count")["total_rub"].mean().round(0))

items_count
1    16739.0
2    33888.0
3    51248.0
Name: total_rub, dtype: float64


In [7]:
# agg позволяет считать несколько разных агрегаций, как в SQL через запятую

print(orders.groupby("status").agg(
    orders=("order_id", "count"),     # имя_колонки=(откуда, чем считать)
    revenue=("total_rub", "sum"),
    avg_order=("total_rub", "mean"),
).round(0))

           orders    revenue  avg_order
status                                 
cancelled     205    7136750    34813.0
completed    6986  236572460    33864.0
returned      136    5503790    40469.0


In [9]:
# size() считает строки в каждой группе, как value_counts

print(orders.groupby(orders["created_at"].dt.to_period("M")).size())

created_at
2025-06      13
2025-07      45
2025-08      90
2025-09     136
2025-10     230
2025-11     384
2025-12     468
2026-01     576
2026-02     664
2026-03    1025
2026-04    1267
2026-05    2427
2026-06       2
Freq: M, dtype: int64


In [19]:
# loc[source] позволяет вывести только нужную/нужные строки

source = 'organic'
res = (users['source'].value_counts(normalize=True) * 100).round(1).loc[source]
print(res)

25.0


# Объединение таблиц

In [21]:
# merge аналоги join

merged = orders.merge(users[["user_id", "country"]], on="user_id")

# проверка строк и столбцов датафрейма
print(f"после: {merged.shape}")

print(merged[["order_id", "user_id", "total_rub", "status", "country"]].head())

после: (7327, 8)
  order_id  user_id  total_rub     status country
0  o_00001  u_00002      25150  completed      RU
1  o_00002  u_00003      31650  completed      RU
2  o_00003  u_00009      13900  completed      UA
3  o_00004  u_00010        690  completed      RU
4  o_00005  u_00013       2100  completed      US


In [23]:
# Сначала фильтруем ранее сджойненный датафрейм, потом считает группировки

completed = merged[merged["status"] == "completed"]
revenue = completed.groupby("country")["total_rub"].sum().sort_values(ascending=False)
print(revenue)

country
RU    118048670
KZ     36474780
BY     22779310
UA     12280270
US     12220350
DE     10534790
PL      7401090
TR      7242150
AM      5434320
GE      4156730
Name: total_rub, dtype: int64


In [24]:
# Перевод результата в миллионы

print((revenue / 1e6).round(1))

country
RU    118.0
KZ     36.5
BY     22.8
UA     12.3
US     12.2
DE     10.5
PL      7.4
TR      7.2
AM      5.4
GE      4.2
Name: total_rub, dtype: float64


In [25]:
# Аналог left join

left = users.merge(orders[["user_id", "order_id"]], on="user_id", how="left")

# Проверка сколько строк с null в конкретном столбце
print(f"пользователей без заказов: {left['order_id'].isna().sum()}")

пользователей без заказов: 2303


In [27]:
# nunique аналог distinct count

buyers = orders["user_id"].nunique()
print(f"покупателей: {buyers} из {len(users)} ({buyers / len(users):.1%})")

покупателей: 3697 из 6000 (61.6%)


# Работа с датами

In [30]:
print("месяц:", orders["created_at"].dt.month.head(3).tolist())

print("год:", orders["created_at"].dt.year.head(3).tolist())

print("день недели:", orders["created_at"].dt.day_name().head(3).tolist())


print("Период год-месяц:", orders["created_at"].dt.to_period('M').head(3).tolist())

print("Период год:", orders["created_at"].dt.to_period('Y').head(3).tolist())

print("Период день", orders["created_at"].dt.to_period('D').head(3).tolist())


месяц: [12, 5, 11]
год: [2025, 2026, 2025]
день недели: ['Monday', 'Saturday', 'Friday']
Период год-месяц: [Period('2025-12', 'M'), Period('2026-05', 'M'), Period('2025-11', 'M')]
Период год: [Period('2025', 'Y-DEC'), Period('2026', 'Y-DEC'), Period('2025', 'Y-DEC')]
Период день [Period('2025-12-22', 'D'), Period('2026-05-09', 'D'), Period('2025-11-21', 'D')]


In [32]:
# resample заполнит нулями столбец, если по этому значению нет данных
# шаг сетки задаётся строкой: "D" дни, "W" недели, "ME" месяцы.

daily = orders.set_index("created_at").resample("D").size()

# внутри loc можно указывать диапазон как в списке
print(daily.loc["2025-11-24":"2025-12-03"])

created_at
2025-11-24     9
2025-11-25    10
2025-11-26    15
2025-11-27    17
2025-11-28    25
2025-11-29    26
2025-11-30    25
2025-12-01    17
2025-12-02    10
2025-12-03    13
Freq: D, dtype: int64


In [34]:
# выручка по месяцам в миллионах

completed = orders[orders["status"] == "completed"]
monthly = completed.set_index("created_at")["total_rub"].resample("ME").sum()
print((monthly / 1e6).round(1))

created_at
2025-06-30     0.3
2025-07-31     1.1
2025-08-31     3.4
2025-09-30     4.3
2025-10-31     8.9
2025-11-30    12.5
2025-12-31    15.5
2026-01-31    18.0
2026-02-28    20.7
2026-03-31    31.2
2026-04-30    40.6
2026-05-31    80.0
2026-06-30     0.0
Freq: ME, Name: total_rub, dtype: float64


In [35]:
# Сколько заказов приходится на сто сессий каждый месяц?

completed = orders[orders["status"] == "completed"]
monthly = completed.set_index("created_at")["total_rub"].resample("ME").sum()
print((monthly / 1e6).round(1))

created_at
2025-06-30     0.3
2025-07-31     1.1
2025-08-31     3.4
2025-09-30     4.3
2025-10-31     8.9
2025-11-30    12.5
2025-12-31    15.5
2026-01-31    18.0
2026-02-28    20.7
2026-03-31    31.2
2026-04-30    40.6
2026-05-31    80.0
2026-06-30     0.0
Freq: ME, Name: total_rub, dtype: float64


# Основные метрики

In [2]:
# Активные пользователи за май (просто те, кто посещал сервис в мае)
may_visit = sessions[
    (sessions["started_at"] >= "2026-05-01") & (sessions["started_at"] < "2026-06-01")
]["user_id"].nunique()

# Активные пользователи за май (те, кто что-то в мае купил)
may_buy = orders[
    (orders["created_at"] >= "2026-05-01") & (orders["created_at"] < "2026-06-01")
]["user_id"].nunique()

print(f"по визитам: {may_visit}, по покупкам: {may_buy}")

по визитам: 4822, по покупкам: 1610


In [3]:
# DAU (daily active users) — число уникальных пользователей за день.
 
# Ключевое слово «уникальных»: пользователь с тремя сессиями за день это один активный пользователь, 
# поэтому nunique, а не подсчёт сессий.

dau = sessions.set_index("started_at").resample("D")["user_id"].nunique()
print(dau.loc["2026-05-25":"2026-05-31"])

started_at
2026-05-25    771
2026-05-26    732
2026-05-27    739
2026-05-28    762
2026-05-29    798
2026-05-30    830
2026-05-31    827
Freq: D, Name: user_id, dtype: int64


In [4]:
# Тоже самое с другой гранулярностью

s = sessions.set_index("started_at")
print(s.resample("W")["user_id"].nunique().tail(4))

print(s.resample("ME")["user_id"].nunique().tail(4))

started_at
2026-05-10    2308
2026-05-17    2433
2026-05-24    2596
2026-05-31    2794
Freq: W-SUN, Name: user_id, dtype: int64
started_at
2026-02-28    2556
2026-03-31    3294
2026-04-30    3970
2026-05-31    4822
Freq: ME, Name: user_id, dtype: int64


In [10]:
# Sticky_factor (липкость)

# Сами по себе DAU и MAU растут у любого продукта с растущей базой — мы это проходили. 
# Их отношение интереснее: средний DAU месяца, делённый на MAU, показывает, 
# какую долю дней месяца типичный пользователь проводит с продуктом.

dau = sessions.set_index("started_at").resample("D")["user_id"].nunique()
mau = s.resample("ME")["user_id"].nunique()
sticky_factor = (dau.resample("ME").mean() / mau * 100).round(1)
print(sticky_factor)

started_at
2025-06-30     4.7
2025-07-31     5.5
2025-08-31     5.4
2025-09-30     6.1
2025-10-31     6.0
2025-11-30     6.5
2025-12-31     6.6
2026-01-31     7.2
2026-02-28     7.6
2026-03-31     8.4
2026-04-30     9.9
2026-05-31    12.9
Freq: ME, Name: user_id, dtype: float64


In [11]:
# Конверсия сессии в покупку

print(f"{(sessions['reached_step'] == 'purchase').mean():.1%}")

10.7%


In [13]:
# Пример расчет разных сегментов воронки конверсий

reached_cart = sessions["reached_step"].isin(["cart", "checkout", "purchase"])
reached_checkout = sessions["reached_step"].isin(["checkout", "purchase"])
reached_purchase = sessions["reached_step"] == "purchase"

print(f"корзина → оформление: {reached_checkout.sum() / reached_cart.sum():.1%}")

print(f"оформление → покупка: {reached_purchase.sum() / reached_checkout.sum():.1%}")

корзина → оформление: 63.0%
оформление → покупка: 79.5%


In [16]:
first_order = orders.groupby("user_id")["created_at"].min().rename("first_order")
u = users.merge(first_order, on="user_id", how="left")
u["days_to_purchase"] = (u["first_order"] - u["signup_date"]).dt.days

print(f"за всё время: {u['first_order'].notna().mean():.1%}")

print(f"за первые 14 дней: {(u['days_to_purchase'] <= 14).mean():.1%}")

# Половина покупателей делает первый заказ позже 36-го дня
print(f"медиана дней до первой покупки: {u['days_to_purchase'].median():.0f}")

за всё время: 61.6%
за первые 14 дней: 18.5%
медиана дней до первой покупки: 36


# Средний чек

In [17]:
# AOV это выручка, делённая на число заказов, то есть обычное среднее по суммам.

completed = orders[orders["status"] == "completed"]

print(f"средний чек: {completed['total_rub'].mean():.0f}")

print(f"медианный чек: {completed['total_rub'].median():.0f}")

средний чек: 33864
медианный чек: 21800


In [18]:
# quantile отвечает на вопрос «дешевле какой суммы лежит такая-то доля заказов»


print(completed["total_rub"].quantile([0.25, 0.5, 0.75, 0.9, 0.99]).round(0))

0.25      8478.0
0.50     21800.0
0.75     46812.0
0.90     89800.0
0.99    145100.0
Name: total_rub, dtype: float64


In [19]:
t = completed["total_rub"]
threshold = t.quantile(0.9)
share = t[t >= threshold].sum() / t.sum()
print(f"топ-10% заказов (дороже {threshold:.0f}) дают {share:.1%} выручки")

топ-10% заказов (дороже 89800) дают 33.6% выручки


# ARPU/ARPPU

In [20]:
may = completed[
    (completed["created_at"] >= "2026-05-01") & (completed["created_at"] < "2026-06-01")
]

may_rev = may["total_rub"].sum()

mau = sessions[
    (sessions["started_at"] >= "2026-05-01") & (sessions["started_at"] < "2026-06-01")
]["user_id"].nunique()

print(f"ARPU:  {may_rev / mau:.0f}")

print(f"ARPPU: {may_rev / may['user_id'].nunique():.0f}")

ARPU:  16582
ARPPU: 51521


In [21]:


def arpu_parts(ym):
    s = sessions[sessions["started_at"].dt.to_period("M").astype(str) == ym]
    
    completed = orders[orders["status"] == "completed"]
    c = completed[completed["created_at"].dt.to_period("M").astype(str) == ym]
    
    mau = s["user_id"].nunique()
    print(f"{ym}: визитов на активного {len(s) / mau:.2f} "
          f"× конверсия {len(c) / len(s):.1%} × чек {c['total_rub'].mean():.0f} "
          f"= {c['total_rub'].sum() / mau:.0f}")

arpu_parts("2026-03")

arpu_parts("2026-05")

2026-03: визитов на активного 2.76 × конверсия 10.7% × чек 32045 = 9466
2026-05: визитов на активного 5.03 × конверсия 9.5% × чек 34585 = 16582


In [24]:
# LTV (lifetime value) это выручка, которую человек приносит за всё время с продуктом.
# Самый простой расчет

# Честную LTV считают по группам пользователей одного возраста

completed = orders[orders["status"] == "completed"]
print(f"на зарегистрированного: {completed['total_rub'].sum() / len(users):.0f}")

print(f"на покупателя: {completed['total_rub'].sum() / completed['user_id'].nunique():.0f}")

на зарегистрированного: 39429
на покупателя: 65678


# Когорты

In [5]:
# Когорта это группа пользователей, объединённых моментом одного и того же стартового события.
# Ключевой трюк когортного анализа: время отсчитывается не по календарю, а от старта когорты. 
# Не «что было в апреле», а «что было на 14-й день жизни пользователя». Так группы разных возрастов становятся сравнимыми


s = sessions.merge(users[["user_id", "signup_date"]], on="user_id")
s["age_days"] = (s["started_at"] - s["signup_date"]).dt.days # вычисление возраста аккаунта на момент сессии


# Оставляются только те сессии, которые произошли в первые 14 дней (включительно) с момента регистрации пользователя. 
# .copy() используется, чтобы избежать предупреждений pandas при дальнейших изменениях этого датафрейма.
early = s[s["age_days"] <= 14].copy()
# Создается столбец cohort, который содержит месяц регистрации пользователя
early["cohort"] = early["signup_date"].dt.to_period("M")

# Здесь мы берем исходную таблицу users, группируем её по месяцу регистрации и считаем общее количество уникальных пользователей, 
# зарегистрировавшихся в каждом месяце. Результат — это Series, где индекс — это месяц, а значение — число пользователей.
cohort_size = users.groupby(users["signup_date"].dt.to_period("M"))["user_id"].count()

# early.groupby("cohort")["session_id"].count() считает общее количество ранних сессий в каждой месячной когорте.
# Это число делится на cohort_size (общее число пользователей в этой когорте).
print((early.groupby("cohort")["session_id"].count() / cohort_size).round(2))

cohort
2025-06    0.48
2025-07    0.63
2025-08    0.57
2025-09    0.71
2025-10    0.78
2025-11    0.90
2025-12    0.98
2026-01    1.29
2026-02    1.68
2026-03    2.29
2026-04    3.94
2026-05    9.27
Freq: M, dtype: float64


In [9]:
# Какая доля визитов кончается покупкой, в тех же самых когортах

# Когорты, чьи первые недели пришлись на апрель и позже, 
# конвертируют ранний визит примерно на пятую часть хуже.

# Активность когорт растёт (это объясняет «здоровые» средние второй фазы), а отдача с визита у новых когорт упала (это трещина из тикета)
early["is_purchase"] = early["reached_step"] == "purchase"
print((early.groupby("cohort")["is_purchase"].mean() * 100).round(1))

cohort
2025-06    10.7
2025-07    12.9
2025-08     8.4
2025-09     9.1
2025-10     9.0
2025-11    15.7
2025-12    11.1
2026-01    12.3
2026-02    12.7
2026-03    10.4
2026-04    10.3
2026-05     9.7
Freq: M, Name: is_purchase, dtype: float64


In [10]:
# Три правила когортного анализа

# Фиксируй окно. Сравнивать когорты можно только по одинаковому отрезку жизни: первые 14 дней против первых 14 дней, а не «всё, что успело накопиться».
# Помечай несозревшие когорты. У 310 из 703 майских пользователей 14-й день жизни ещё не наступил: их окно обрезано справа, и значение когорты занижено. Такие точки либо исключают, либо явно помечают.
# Подписывай размер когорты. Маленькие когорты шумят: летние 8.4% и 15.7% это не события, а статистическая рябь на паре сотен человек.

# Retention-матрица

In [11]:
# Retention-матрица отвечает на вопрос, который задаёт себе каждый продукт: возвращаются ли к нам люди. 
# Строки в ней это когорты, столбцы это месяцы жизни когорты, в ячейке доля когорты, дожившая до этого месяца.

In [13]:
# Этот код реализует классический когортный анализ удержания (Retention Analysis). 
# Он строит знаменитую "треугольную" таблицу (Retention Triangle), которая показывает, какой процент пользователей, 
# зарегистрировавшихся в определенном месяце, вернулся в приложение в последующие месяцы (0-й месяц, 1-й месяц, 2-й месяц и так далее


users["cohort"] = users["signup_date"].dt.to_period("M") # Теперь мы знаем, к какому месячному "призыву" (когорте) относится каждый пользователь.
s = sessions.merge(users[["user_id", "cohort"]], on="user_id") # Теперь каждая строка-сессия "знает", в каком месяце зарегистрировался её автор.
s["month"] = s["started_at"].dt.to_period("M") # Мы создаем столбец month, который показывает месяц, когда произошла конкретная сессия.

# Это ключевой шаг. Мы вычитаем месяц регистрации (cohort) из месяца сессии (month).
# Если пользователь зарегистрировался в Январе (2023-01) и сделал сессию в Январе (2023-01), разница равна 0.
# Если сессия была в Феврале (2023-02), разница равна 1.
# Если в Марте (2023-03), разница равна 2.
# Метод .apply(lambda d: d.n) извлекает из полученного объекта разницы периодов (PeriodOffset) чистое целое число. 
# Столбец m теперь показывает: "Сколько полных месяцев прошло с момента регистрации".
s["m"] = (s["month"] - s["cohort"]).apply(lambda d: d.n)

# groupby(["cohort", "m"]): мы группируем данные по месяцу регистрации и по номеру месяца с момента регистрации.
# ["user_id"].nunique(): мы считаем количество уникальных пользователей (это критически важно! В предыдущем вашем коде был count(), который считал сессии. Здесь nunique() считает именно людей, что правильно для Retention).
# .unstack(): эта функция берет внутренний уровень группировки (m) и превращает его в столбцы.
active = s.groupby(["cohort", "m"])["user_id"].nunique().unstack()

# Мы считаем, сколько всего пользователей зарегистрировалось в каждом месяце. 
# Это Series, где индекс — это месяц (cohort), а значение — общее число людей.
cohort_size = users.groupby("cohort")["user_id"].count()

# Мы делим таблицу active на Series cohort_size.
# Параметр axis=0 здесь критически важен. Он говорит pandas: 
# "Бери каждую строку таблицы active и дели все её значения на число из cohort_size, которое соответствует индексу (названию когорты) этой строки".
# * 100: превращает долю в проценты.
retention = (active.div(cohort_size, axis=0) * 100).round(1)
print(retention.iloc[:, :7])

m            0     1     2     3     4     5     6
cohort                                            
2025-06   31.9  53.5  50.0  50.0  60.2  57.5  57.9
2025-07   38.9  59.8  62.3  65.4  60.4  62.6  62.6
2025-08   33.5  58.4  61.0  63.2  63.5  52.1  54.9
2025-09   41.5  63.7  66.1  66.8  67.3  63.5  65.9
2025-10   44.3  68.2  69.7  65.6  67.0  67.6  69.5
2025-11   44.3  75.4  72.9  69.7  73.6  72.9  74.3
2025-12   51.0  77.1  72.0  77.6  74.4  75.0   NaN
2026-01   58.6  78.6  80.3  80.9  79.3   NaN   NaN
2026-02   63.8  87.9  85.7  85.2   NaN   NaN   NaN
2026-03   72.1  90.4  93.0   NaN   NaN   NaN   NaN
2026-04   81.1  96.2   NaN   NaN   NaN   NaN   NaN
2026-05  100.0   NaN   NaN   NaN   NaN   NaN   NaN


In [15]:
# Как читать

# Строка → «Жизненный путь одного призыва»
# Что это: Берёшь одну строку (одну когорту) и читаешь её слева направо. Это как смотреть документальный фильм про конкретный «призыв» пользователей — от их первого дня до сегодняшнего.
# Что ищем: Как менялась активность этой когорты со временем? Где она «осела» (вышла на плато)? Есть ли провалы или всплески?
# Зачем это нужно: Понять, как ведёт себя конкретный призыв. Если все когорты ведут себя одинаково (например, все выходят на плато 60%) — это здоровый паттерн, продукт работает стабильно.

# Столбец → «Сравнение призывов в одном возрасте»
# Что это: Берёшь один столбец (один возраст, например, m=1) и читаешь его сверху вниз. Это как сделать «фотографию» всех когорт в один и тот же момент их жизни — например, через месяц после регистрации.
# Что ищем: Становится ли продукт лучше со временем? Каждый новый призыв «живучее» предыдущего или хуже?
# Зачем это нужно: Понять, улучшается ли продукт. Если столбцы растут сверху вниз — продукт становится лучше, пользователи приходят более «качественные» или онбординг работает эффективнее. 
# Если столбцы падают — что-то сломалось.

# Диагональ → «Один календарный месяц во всех когортах»
# Что это: Идёшь по диагонали сверху-вниз. Это как взять «фотографию» всего продукта в один конкретный календарный месяц (например, в сентябре 2025) и посмотреть, 
# как в этот месяц вели себя все когорты — каждая в своём возрасте.
# Зачем это нужно: Найти внешние события, которые повлияли на всех пользователей одновременно. 
# Если провал идёт по диагонали — значит, проблема была в продукте в конкретный месяц (баг, изменение политики, сезонность). 
# Если провал идёт по строке — значит, проблема была в конкретном призыве (плохой маркетинг, некачественные пользователи).




In [16]:
# По строкам: Когорты не «умирают», а набирают активность со временем. Это нетипично для классического ретеншена (обычно значения падают). 
# Возможно, у тебя продукт с отложенным эффектом (обучение, подписка) или пользователи возвращаются после перерыва.
# По столбцам: Каждый новый призыв значительно активнее предыдущего. Это отличный знак — продукт улучшается.
# По диагоналям: Нет явных провалов, которые шли бы по диагоналям. Значит, не было катастрофических событий, которые бы одновременно уронили все когорты.

# LTV-когорты

In [ ]:
# LTV-кривая когорты отвечает на вопрос «сколько в среднем принёс один пользователь когорты к месяцу жизни m».

users["cohort"] = users["signup_date"].dt.to_period("M")
completed = orders[orders["status"] == "completed"]

c = completed.merge(users[["user_id", "cohort"]], on="user_id")
c["month"] = c["created_at"].dt.to_period("M")
c = c[c["month"] <= "2026-05"] # Фильтр c["month"] <= "2026-05" отрезает пограничный июньский хвост из двух заказов, чтобы правый край матрицы не засоряли копеечные ячейки.
c["m"] = (c["month"] - c["cohort"]).apply(lambda d: d.n)

cohort_size = users.groupby("cohort")["user_id"].count()
rev = c.groupby(["cohort", "m"])["total_rub"].sum().unstack()
ltv = rev.cumsum(axis=1).div(cohort_size, axis=0).round(0)
print(ltv.iloc[:, :7])

m              0        1        2        3        4        5        6
cohort                                                                
2025-06   1345.0   3744.0   8945.0  10762.0  16159.0  19153.0  25573.0
2025-07   1541.0   6397.0  10345.0  15960.0  21590.0  25563.0  28775.0
2025-08   1386.0   4380.0   9775.0  14497.0  19150.0  22210.0  25765.0
2025-09   3203.0   8922.0  13334.0  19591.0  25831.0  30508.0  36052.0
2025-10   2510.0   9460.0  15575.0  21934.0  26504.0  31007.0  37341.0
2025-11   5089.0  11028.0  17775.0  23808.0  29861.0  35682.0  42590.0
2025-12   3323.0  10584.0  16993.0  25409.0  29714.0  37243.0      NaN
2026-01   4156.0  11556.0  19009.0  26536.0  35084.0      NaN      NaN
2026-02   6595.0  19234.0  29526.0  42712.0      NaN      NaN      NaN
2026-03   6776.0  22851.0  37585.0      NaN      NaN      NaN      NaN
2026-04  11441.0  40267.0      NaN      NaN      NaN      NaN      NaN
2026-05  33699.0      NaN      NaN      NaN      NaN      NaN      NaN


In [26]:
# Мы фильтруем таблицу заказов, оставляя только те, у которых статус "completed". 
# Отмененные или ожидающие оплаты заказы не приносят реальной выручки, поэтому для расчета LTV они не нужны.
completed = orders[orders["status"] == "completed"]

# К успешным заказам мы присоединяем из таблицы users три важных столбца:
# user_id (ключ для связи)
# cohort (месяц регистрации пользователя)
# source (источник трафика, откуда пришел пользователь, например, "organic", "facebook_ads", "email").
src = completed.merge(users[["user_id", "cohort", "source"]], on="user_id")
src["m"] = (src["created_at"].dt.to_period("M") - src["cohort"]).apply(lambda d: d.n)

# Мы берем только тех пользователей, которые зарегистрировались в феврале 2026 года или раньше.
# Зачем? Чтобы посчитать выручку за 3 месяца (m <= 2), у когорты должно быть минимум 3 месяца "жизни" в прошлом
mature = users[users["cohort"] <= "2026-02"]


# Из таблицы с заказами (src) мы оставляем только те строки, которые удовлетворяют двум условиям:
# Заказ был сделан в первые 3 месяца жизни пользователя (m <= 2, то есть месяцы 0, 1 и 2).
# Пользователь принадлежит к "зрелой" когорте (чтобы данные были сопоставимы).
early_rev = src[(src["m"] <= 2) & (src["cohort"] <= "2026-02")]

print(early_rev.groupby("source")["total_rub"].sum())
print(mature.groupby("source")["user_id"].count())
# На самом деле, здесь делится не датафрейм на датафрейм, а Series (одномерный столбец с индексом) на Series.
# Вся магия происходит благодаря механизму, который в pandas называется Index Alignment (выравнивание по индексу). 
# Pandas не делит "первую строку на первую строку" наугад. 
# Он смотрит на названия индексов (в данном случае — на названия источников трафика) и делит значения только там, где названия совпадают.

# Если индексы не соответствуют, Pandas не выдаст ошибку. 
# Он увидит, что для "email" нет пары в левой части, и автоматически подставит значение NaN (Not a Number)
ltv3 = (
    early_rev.groupby("source")["total_rub"].sum()
    / mature.groupby("source")["user_id"].count()
).round(0).sort_values(ascending=False)
print(ltv3)

source
email           8873420
organic        15947690
paid_search    14148110
paid_social    18696130
referral        7748880
Name: total_rub, dtype: int64
source
email           406
organic         963
paid_search     784
paid_social    1363
referral        382
Name: user_id, dtype: int64
source
email          21856.0
referral       20285.0
paid_search    18046.0
organic        16560.0
paid_social    13717.0
dtype: float64
